In [5]:
import pandas as pd

# 타당성 확인을 거친 절대 경로 데이터 주입
file_path = r"D:\ko-medscribe-llm\ambient-doc\aci-bench\data\challenge_data\train.csv"

# 데이터 로드
df = pd.read_csv(file_path)

# 첫 번째 샘플 추출 및 검증
print(f"데이터 로드 성공! 총 샘플 수: {len(df)}개")
first_sample = df.iloc[0]
print("\n--- 첫 번째 샘플 데이터 ---")
print(first_sample)

데이터 로드 성공! 총 샘플 수: 67개

--- 첫 번째 샘플 데이터 ---
dataset                                                virtassist
encounter_id                                               D2N001
dialogue        [doctor] hi , martha . how are you ?\n[patient...
note            CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
Name: 0, dtype: object


In [6]:
import pandas as pd

file_path = r"D:\ko-medscribe-llm\ambient-doc\aci-bench\data\challenge_data\train.csv"
df = pd.read_csv(file_path)

print("="*50)
print(f"데이터 로드 성공! (총 샘플 수: {len(df)}개)")
print("="*50)
print(df.info())  # 컬럼 구조 출력
print("="*50)

데이터 로드 성공! (총 샘플 수: 67개)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   dataset       67 non-null     object
 1   encounter_id  67 non-null     object
 2   dialogue      67 non-null     object
 3   note          67 non-null     object
dtypes: object(4)
memory usage: 2.2+ KB
None


In [7]:
import pandas as pd

file_path = r"D:\ko-medscribe-llm\ambient-doc\aci-bench\data\challenge_data\train.csv"
df = pd.read_csv(file_path)

print("="*60)
print("SUCCESS: DATA LOAD")
print("="*60)

print("\n[DIALOGUE SAMPLE]")
print(df['dialogue'].iloc[0][:200])

print("\n[NOTE SAMPLE]")
print(df['note'].iloc[0][:200])
print("="*60)

df['dialogue_len'] = df['dialogue'].str.len()
df['note_len'] = df['note'].str.len()

print("\n[DIALOGUE STATS]")
print(df['dialogue_len'].describe())

print("\n[NOTE STATS]")
print(df['note_len'].describe())
print("="*60)

SUCCESS: DATA LOAD

[DIALOGUE SAMPLE]
[doctor] hi , martha . how are you ?
[patient] i'm doing okay . how are you ?
[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?


[NOTE SAMPLE]
CHIEF COMPLAINT

Annual exam.

HISTORY OF PRESENT ILLNESS

Martha Collins is a 50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension who

[DIALOGUE STATS]
count       67.000000
mean      6443.746269
std       1976.513431
min       2938.000000
25%       5051.000000
50%       6247.000000
75%       7635.500000
max      14186.000000
Name: dialogue_len, dtype: float64

[NOTE STATS]
count      67.000000
mean     2649.283582
std       815.526021
min       852.000000
25%      2274.000000
50%      2537.000000
75%      3076.000000
max      5712.000000
Name: note_len, dtype: float64


In [8]:
import pandas as pd
import json

file_path = r"D:\ko-medscribe-llm\ambient-doc\aci-bench\data\challenge_data\train.csv"
df = pd.read_csv(file_path)

# LLM 주입용 프롬프트 템플릿 정의
def 데이터_가공(row):
    system_prompt = "You are an expert clinical AI assistant. Based on the following conversation between a doctor and a patient, generate a structured clinical note including CHIEF COMPLAINT and HISTORY OF PRESENT ILLNESS."
    user_prompt = f"Conversation:\n{row['dialogue']}\n\nGenerate Clinical Note:"
    
    # OpenAI/Llama3 메시지 포맷에 맞춘 가공
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": row['note']}
        ]
    }

# 1번 샘플 변환 테스트
sample_formatted = 데이터_가공(df.iloc[0])

print("="*60)
print("[단계 4] LLM 학습 포맷 변환 검증")
print("="*60)
print(json.dumps(sample_formatted, indent=2)[:800] + "\n... (이하 생략) ...")
print("="*60)

[단계 4] LLM 학습 포맷 변환 검증
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert clinical AI assistant. Based on the following conversation between a doctor and a patient, generate a structured clinical note including CHIEF COMPLAINT and HISTORY OF PRESENT ILLNESS."
    },
    {
      "role": "user",
      "content": "Conversation:\n[doctor] hi , martha . how are you ?\n[patient] i'm doing okay . how are you ?\n[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?\n[patient] okay .\n[doctor] martha is a 50-year-old female with a past medical history significant for congestive heart failure , depression and hypertension who presents for her annual exam . so , martha , it's been a year since i've seen you . how are you do
... (이하 생략) ...


In [10]:
import pandas as pd
import json
import os

# 1. 데이터 로드
file_path = r"D:\ko-medscribe-llm\ambient-doc\aci-bench\data\challenge_data\train.csv"
df = pd.read_csv(file_path)

# 2. 출력 경로 설정 (현재 파이썬이 실행 중인 디렉토리에 바로 저장)
output_dir = os.getcwd()
output_file = os.path.join(output_dir, "train_llm.jsonl")

print("="*60)
print("START: PROCESSING ALL 67 SAMPLES")
print("="*60)

# 3. JSONL 파일 쓰기
count = 0
with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        system_prompt = "You are an expert clinical AI assistant. Based on the following conversation between a doctor and a patient, generate a structured clinical note including CHIEF COMPLAINT and HISTORY OF PRESENT ILLNESS."
        user_prompt = f"Conversation:\n{row['dialogue']}\n\nGenerate Clinical Note:"
        
        data_structure = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": row['note']}
            ]
        }
        
        f.write(json.dumps(data_structure, ensure_ascii=False) + "\n")
        count += 1

print(f"SUCCESS: {count}개의 샘플이 변환 완료되었습니다.")
print(f"SAVED AT: {output_file}")
print("="*60)

# 4. 파일 크기 검증
if os.path.exists(output_file):
    file_size_kb = os.path.getsize(output_file) / 1024
    print(f"생성된 파일 크기: {file_size_kb:.2f} KB")
print("="*60)

START: PROCESSING ALL 67 SAMPLES
SUCCESS: 67개의 샘플이 변환 완료되었습니다.
SAVED AT: d:\ko-medscribe-llm\ko-medscribe-llm\notebooks\train_llm.jsonl
생성된 파일 크기: 626.40 KB


In [1]:
import json
import os
import requests

# [수정] 파일이 확실하게 존재하는 D드라이브 절대 경로로 강제 고정
jsonl_file = r"D:\ko-medscribe-llm\train_llm.jsonl"

if not os.path.exists(jsonl_file):
    # 만약 위 경로에도 없다면 다른 예상 경로 확인
    jsonl_file = r"D:\ko-medscribe-llm\ko-medscribe-llm\notebooks\train_llm.jsonl"

if not os.path.exists(jsonl_file):
    print(f"Error: 데이터셋 파일을 찾을 수 없습니다. 경로를 확인하세요.")
    exit()

all_samples = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        all_samples.append(json.loads(line))

# 2. Few-shot 조립 (0, 1번 예시 주입 / 2번 테스트)
few_shot_examples = all_samples[0:2]
test_sample = all_samples[2]

messages = []
messages.append(few_shot_examples[0]["messages"][0]) # System Prompt

for sample in few_shot_examples:
    messages.append(sample["messages"][1]) # User 예시
    messages.append(sample["messages"][2]) # Assistant 정답 예시

messages.append(test_sample["messages"][1]) # 실제 테스트할 대화

# 3. Ollama 로컬 API 호출
ollama_url = "http://localhost:11434/api/chat"
payload = {
    "model": "llama3",
    "messages": messages,
    "stream": False
}

print("="*60)
print(f"LOADED FILE: {jsonl_file}")
print("로컬 Ollama LLM 추론 시작 (최초 가동 시 1~2분 소요)...")
print("="*60)

try:
    response = requests.post(ollama_url, json=payload, timeout=90)
    if response.status_code == 200:
        result = response.json()
        generated_note = result['message']['content']
        
        print("\n" + "="*60)
        print("▲ [AI가 생성한 임상 노트 (Few-Shot 결과)]")
        print("="*60)
        print(generated_note)
        
        print("\n" + "="*60)
        print("▲ [의사가 작성한 원본 정답 (Ground Truth)]")
        print("="*60)
        print(test_sample["messages"][2]["content"])
        print("="*60)
    else:
        print(f"Ollama 서버 응답 에러. 상태 코드: {response.status_code}")
except Exception as e:
    print(f"에러 발생: {e}")

LOADED FILE: D:\ko-medscribe-llm\ko-medscribe-llm\notebooks\train_llm.jsonl
로컬 Ollama LLM 추론 시작 (최초 가동 시 1~2분 소요)...

▲ [AI가 생성한 임상 노트 (Few-Shot 결과)]
CHIEF COMPLAINT

Back pain.

HISTORY OF PRESENT ILLNESS

John is a 61-year-old male with a past medical history significant for kidney stones, migraines, and reflux. He presents today with complaints of back pain.

The patient reports that the pain started on the right side of his back but has shifted to both sides over time. The pain has been constant for the last 48 hours and is quite severe. He has tried taking Tylenol for the pain, but it does not seem to provide much relief.

The patient also endorses blood in his urine, which he believes may be related to the kidney stones. He experiences dizziness and lightheadedness when exerting himself, which is thought to be related to the back pain.

The patient has been compliant with taking his Imitrex medication for migraines and reports no issues with those. His acid reflux has also been w